In [ ]:
## Version 2
# %% [markdown]
# # INFO-F-422 Project: EMG-based Hand Pose Estimation (Guided Gestures)
#
# **Team:** [Your Team Name/Number]
# **Members:** [Your Names]
#
# This notebook implements baseline and neural network approaches for predicting hand joint angles from sEMG signals using the guided gestures dataset.

# %%
# Core Libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import time
import pickle # To save models/results if needed
from contextlib import contextmanager

# Scikit-learn Modules
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler # Key for preprocessing features
from sklearn.impute import SimpleImputer # Example if needed, but unlikely for features
from sklearn.model_selection import GroupKFold, cross_validate, train_test_split, GridSearchCV
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import mean_squared_error, make_scorer, r2_score
from sklearn.linear_model import Ridge, Lasso # Added Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn import set_config # For metadata routing

# PyTorch Modules (for Objective 5)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Configure visualizations
%matplotlib inline
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Define a timing context manager for convenience
@contextmanager
def timer(label: str):
    start = time.time()
    print(f"{label}...")
    yield
    end = time.time()
    print(f"{label} done in {end - start:.2f} seconds.\n")

# Configure metadata routing (needed for SFS and potentially CV with params)
set_config(enable_metadata_routing=True)
print(f"Scikit-learn metadata routing enabled: {set_config(True)}") # Confirm it's set

# Set device for PyTorch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using PyTorch device: {device}")



# %% [markdown]
# ## 1. Load Data
#
# Load the guided gestures dataset files. (Code identical to previous version)

# %%
# File paths
X_TRAIN_VAL_PATH = "guided_dataset_X.npy"
Y_TRAIN_VAL_PATH = "guided_dataset_y.npy"
X_TEST_PATH = "guided_testset_X.npy"

# Load the training/validation data
X_train_val_raw = np.load(X_TRAIN_VAL_PATH)
y_train_val_raw = np.load(Y_TRAIN_VAL_PATH)

# Load the test data structure description (we need shape info)
X_test_raw_info = np.load(X_TEST_PATH, mmap_mode='r')

# Print shapes to verify
print("Raw Training/Validation Data Shapes:")
print(f"X_train_val_raw: {X_train_val_raw.shape}")
print(f"y_train_val_raw: {y_train_val_raw.shape}")
print("\nRaw Test Data Shape (for info):")
print(f"X_test_raw_info: {X_test_raw_info.shape}")

# Constants derived from data
N_SESSIONS_TRAIN_VAL = X_train_val_raw.shape[0]
N_ELECTRODES = X_train_val_raw.shape[1]
N_TIME_POINTS_TRAIN_VAL = X_train_val_raw.shape[2]
N_JOINTS = y_train_val_raw.shape[1]

print(f"\nNumber of Training/Validation Sessions: {N_SESSIONS_TRAIN_VAL}")
print(f"Number of Electrodes: {N_ELECTRODES}")
print(f"Number of Time Points per Session: {N_TIME_POINTS_TRAIN_VAL}")
print(f"Number of Joints: {N_JOINTS}")

# %% [markdown]
# ## 2. Signal Filtering (Optional but Applied)
#
# Applying band-pass filtering (20-450 Hz) and DC offset removal to the raw sEMG data before windowing.

# %%
# Example (requires scipy):
from scipy.signal import butter, filtfilt

def butter_bandpass(lowcut, highcut, fs, order=5):
     nyq = 0.5 * fs
     low = lowcut / nyq
     high = highcut / nyq
     b, a = butter(order, [low, high], btype='band')
     return b, a

def bandpass_filter(data, lowcut, highcut, fs, order=5):
     b, a = butter_bandpass(lowcut, highcut, fs, order=order)
     # Apply filter along the time axis (axis=-1)
     y = filtfilt(b, a, data, axis=-1)
     return y

FS = 1024 # Sampling frequency from documentation
LOWCUT = 20
HIGHCUT = 450

with timer("Applying Signal Filtering"):
    X_train_val_filtered = np.zeros_like(X_train_val_raw)
    for i in range(N_SESSIONS_TRAIN_VAL):
        # Remove DC offset (mean) per channel
        dc_removed = X_train_val_raw[i] - np.mean(X_train_val_raw[i], axis=-1, keepdims=True)
        # Apply bandpass filter
        X_train_val_filtered[i] = bandpass_filter(dc_removed, LOWCUT, HIGHCUT, FS)

    X_train_val_to_window = X_train_val_filtered
    print("Applied DC offset removal and bandpass filtering.")



# %% [markdown]
# ## 3. Dataset Preparation: Overlapping Windows (Objective 2)
#
# Segment the filtered sEMG and raw joint angle data into overlapping windows. (Code identical to previous version)

# %%
# Windowing Parameters
WINDOW_SIZE = 500  # k
OVERLAP_PERCENTAGE = 0.50 # Using 50% overlap

step = int(WINDOW_SIZE * (1 - OVERLAP_PERCENTAGE))
if step == 0:
    raise ValueError("Overlap percentage cannot be 100%")
print(f"Window Size (k): {WINDOW_SIZE}")
print(f"Overlap: {OVERLAP_PERCENTAGE*100:.0f}%")
print(f"Step Size: {step}")

# (create_overlapping_windows function definition is the same as before)
def create_overlapping_windows(X_raw, y_raw, window_size, step):
    n_sessions, n_electrodes, n_timepoints = X_raw.shape
    n_joints = y_raw.shape[1]
    X_windowed_list, y_windowed_list, groups_list = [], [], []
    for session_idx in range(n_sessions):
        session_X = X_raw[session_idx]
        session_y = y_raw[session_idx]
        start_indices = np.arange(0, n_timepoints - window_size + 1, step)
        for start_idx in start_indices:
            end_idx = start_idx + window_size
            window_X = session_X[:, start_idx:end_idx]
            target_y = session_y[:, end_idx - 1]
            X_windowed_list.append(window_X)
            y_windowed_list.append(target_y)
            groups_list.append(session_idx)
        print(f"Session {session_idx}: Created {len(start_indices)} windows.")
    return np.stack(X_windowed_list), np.stack(y_windowed_list), np.array(groups_list)


# Create the windowed dataset
with timer("Creating overlapping windows"):
    X_windowed, y_windowed, groups = create_overlapping_windows(
        X_train_val_to_window, y_train_val_raw, WINDOW_SIZE, step
    )

print("\nWindowed Data Shapes:")
print(f"X_windowed: {X_windowed.shape}")
print(f"y_windowed: {y_windowed.shape}")
print(f"Groups: {groups.shape}")
print(f"Unique groups (sessions): {np.unique(groups)}")


# %% [markdown]
# ## 4. Data Preprocessing (Features)
#
# The primary preprocessing step after feature extraction is **normalization/scaling**. We use `StandardScaler` to standardize features by removing the mean and scaling to unit variance. This is crucial for distance-based algorithms, linear models (Ridge, Lasso), and neural networks.
#
# **Missing Values:** The `EMGFeatureExtractor` includes `np.nan_to_num` to handle potential issues arising during feature calculation (e.g., variance of a constant signal is zero, leading to NaN in STD if not handled). We assume the raw EMG or windowed data does not contain NaNs initially.
#
# **Outliers:** We are not applying explicit outlier removal at this stage. `StandardScaler` is somewhat sensitive to outliers, but EMG signals can naturally have spikes. Robust scaling methods (like `RobustScaler`) could be considered if outlier impact is significant.
#
# **Implementation:** `StandardScaler` will be included *within* the Scikit-learn pipelines for baseline models and applied explicitly during the Neural Network's cross-validation loop to prevent data leakage.

# %%
print("Preprocessing Step: StandardScaler will be applied to extracted features within pipelines and NN CV loop.")
# Example: If you wanted to check features *before* scaling (optional)
# temp_extractor = EMGFeatureExtractor()
# temp_features = temp_extractor.fit_transform(X_windowed)
# print(f"Feature shape before scaling: {temp_features.shape}")
# print(f"Example features mean (before scaling): {np.mean(temp_features[:, 0]):.4f}")
# print(f"Example features std (before scaling): {np.std(temp_features[:, 0]):.4f}")
# del temp_extractor, temp_features # Clean up




# %% [markdown]
# ## 5. Custom Feature Extractor (Baseline - Objective 4)
#
# Extracts time-domain features. (Code identical to previous version)

# %%
class EMGFeatureExtractor(BaseEstimator, TransformerMixin):
    """ Extracts time-domain features from EMG windows. (Code identical) """
    def __init__(self, zc_threshold=1e-5, ssc_threshold=1e-5, selected_features=None):
        self.zc_threshold = zc_threshold
        self.ssc_threshold = ssc_threshold
        self.all_feature_names = ['MAV', 'RMS', 'Var', 'STD', 'ZC', 'MPR', 'WL', 'SSC']
        if selected_features is None: self.selected_features = self.all_feature_names
        else:
            if not all(f in self.all_feature_names for f in selected_features):
                raise ValueError(f"Invalid features. Allowed: {self.all_feature_names}")
            self.selected_features = selected_features

    def fit(self, X, y=None): return self

    def transform(self, X):
        n_windows, n_electrodes, window_size = X.shape
        n_selected = len(self.selected_features)
        features = np.zeros((n_windows, n_electrodes * n_selected))
        for i in range(n_windows):
            window_features = []
            for j in range(n_electrodes):
                channel_data = X[i, j, :]
                channel_features = []
                mav = np.mean(np.abs(channel_data))
                rms = np.sqrt(np.mean(channel_data**2))
                var = np.var(channel_data)
                std = np.sqrt(var)
                if 'MAV' in self.selected_features: channel_features.append(mav)
                if 'RMS' in self.selected_features: channel_features.append(rms)
                if 'Var' in self.selected_features: channel_features.append(var)
                if 'STD' in self.selected_features: channel_features.append(std)
                if 'ZC' in self.selected_features:
                    zc_count = 0
                    for k in range(window_size - 1):
                        if ((channel_data[k] > self.zc_threshold and channel_data[k+1] < self.zc_threshold) or \
                           (channel_data[k] < self.zc_threshold and channel_data[k+1] > self.zc_threshold)):
                            zc_count += 1
                    channel_features.append(zc_count)
                if 'MPR' in self.selected_features:
                    threshold_mpr = std if std > 1e-9 else 1e-9
                    mpr = np.mean(np.abs(channel_data) > threshold_mpr)
                    channel_features.append(mpr)
                if 'WL' in self.selected_features:
                    wl = np.sum(np.abs(np.diff(channel_data)))
                    channel_features.append(wl)
                if 'SSC' in self.selected_features:
                    diff_sig = np.diff(channel_data)
                    ssc_count = 0
                    for k in range(len(diff_sig) - 1):
                         if ((diff_sig[k] > self.ssc_threshold and diff_sig[k+1] < -self.ssc_threshold) or \
                            (diff_sig[k] < -self.ssc_threshold and diff_sig[k+1] > self.ssc_threshold)):
                             ssc_count += 1
                    channel_features.append(ssc_count)
                window_features.extend(channel_features)
            features[i, :] = window_features
        features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
        return features

    def get_feature_names_out(self, input_features=None):
        feature_names = []
        for j in range(N_ELECTRODES):
            for feature_name in self.selected_features:
                feature_names.append(f"Elec{j+1}_{feature_name}")
        return feature_names

# Test the feature extractor (on first few windows for speed if needed)
with timer("Testing Feature Extractor"):
     feature_extractor = EMGFeatureExtractor()
     # Use X_windowed directly as input
     X_features = feature_extractor.fit_transform(X_windowed)

print("Feature Matrix Shape:")
print(f"X_features: {X_features.shape}")
n_expected_features = N_ELECTRODES * len(feature_extractor.selected_features)
print(f"Expected number of features per window: {n_expected_features}")
assert X_features.shape[1] == n_expected_features, "Feature dimension mismatch!"
feature_names = feature_extractor.get_feature_names_out()
print(f"\nGenerated {len(feature_names)} feature names. Examples: {feature_names[:5]}")



# %% [markdown]
# ## 6. Cross-Validation Strategy (Objective 3)
#
# Using `GroupKFold` with session IDs as groups (Leave-One-Session-Out). (Code identical to previous version)

# %%
N_SPLITS = N_SESSIONS_TRAIN_VAL # Use Leave-One-Session-Out
cv_strategy = GroupKFold(n_splits=N_SPLITS)
print(f"Using GroupKFold with {N_SPLITS} splits (Leave-One-Session-Out).")

# (plot_cv_indices function definition is the same as before)
def plot_cv_indices(cv, X, y, group, n_splits):
    fig, ax = plt.subplots(1, 1, figsize=(10, 3))
    cmap_data = plt.cm.Paired; cmap_cv = plt.cm.coolwarm
    for i, (train_idx, test_idx) in enumerate(cv.split(X, y, group)):
        indices = np.array([np.nan] * len(X)); indices[test_idx] = 1; indices[train_idx] = 0
        ax.scatter(range(len(indices)), [i + .5] * len(indices), c=indices, marker='_', lw=10, cmap=cmap_cv, vmin=-.2, vmax=1.2)
    ax.scatter(range(len(X)), [i + 1.5] * len(X), c=group, marker='_', lw=10, cmap=cmap_data)
    yticklabels = list(range(n_splits)) + ['Session']
    ax.set(yticks=np.arange(n_splits + 1) + .5, yticklabels=yticklabels, xlabel='Sample Index', ylabel="CV Iteration / Session", ylim=[n_splits + 1.2, -.2], xlim=[0, len(y)])
    ax.set_title('GroupKFold Splits Visualization', fontsize=15); plt.tight_layout(); plt.show()

# Plot for our setup using the generated features X_features
plot_cv_indices(cv_strategy, X_features, y_windowed, groups, N_SPLITS)
# Temporarily disable plotting if it takes too long or isn't needed repeatedly
print("CV Split visualization plot skipped for brevity.")



# %% [markdown]
# ## 7. Baseline Model Training and Evaluation (Objective 4 - Expanded)
#
# We now evaluate Ridge, Lasso, and Random Forest regressors.
# The pipeline includes feature extraction and standardization.

# %%
# Define Performance Metrics (Identical code)
def root_mean_squared_error(y_true, y_pred): return np.sqrt(mean_squared_error(y_true, y_pred))
def normalized_mean_squared_error(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    if y_true.shape != y_pred.shape: raise ValueError("Shape mismatch")
    if y_true.ndim == 1: y_true, y_pred = y_true.reshape(-1, 1), y_pred.reshape(-1, 1)
    numerator = np.sum((y_true - y_pred)**2)
    mean_y_true_per_joint = np.mean(y_true, axis=0)
    denominator = np.sum((y_true - mean_y_true_per_joint)**2)
    return np.inf if denominator == 0 and numerator > 1e-9 else (0.0 if denominator == 0 else numerator / denominator)

scorers = {
    'rmse': make_scorer(root_mean_squared_error, greater_is_better=False),
    'nmse': make_scorer(normalized_mean_squared_error, greater_is_better=False),
    'r2': make_scorer(r2_score)
}

# %%
# Select Models - Adding Lasso
from sklearn.linear_model import Lasso # Ensure import

model_ridge = Ridge(alpha=1.0) # Basic regularization
model_lasso = Lasso(alpha=0.01, max_iter=2000, tol=1e-3) # Lasso needs careful alpha tuning, start low
model_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    n_jobs=-1,
    random_state=42,
    max_features=0.33
)

# Store all baseline models here
baseline_models_to_evaluate = {
    "Ridge": model_ridge,
    "Lasso": model_lasso,
    "RandomForest": model_rf
}

# Store all results here across objectives
all_cv_results = {}

# %%
# Evaluate baseline models using Cross-Validation
print("--- Evaluating Baseline Models (Ridge, Lasso, RF) ---")
for model_name, model in baseline_models_to_evaluate.items():
    with timer(f"Cross-validating {model_name}"):
        pipeline = Pipeline([
            ('feature_extraction', EMGFeatureExtractor()),
            ('scaling', StandardScaler()), # Preprocessing step
            ('regression', model)
        ])

        # Perform cross-validation using params for groups
        scores = cross_validate(
            pipeline,
            X_windowed,         # Input raw windowed data
            y_windowed,         # Target
            cv=cv_strategy,
            scoring=scorers,
            n_jobs=-1,
            return_train_score=False,
            params={'groups': groups} # Pass groups via params
        )
        all_cv_results[model_name] = scores # Store results

# %%
# Display Baseline CV Results
print("\nBaseline Cross-Validation Performance Results:")

results_summary_list = [] # Keep track of summaries
for model_name, scores in all_cv_results.items():
     # Check if model is in the baseline set before printing detailed baseline results
     if model_name in baseline_models_to_evaluate:
        mean_rmse = -np.mean(scores['test_rmse'])
        std_rmse = np.std(scores['test_rmse'])
        mean_nmse = -np.mean(scores['test_nmse'])
        std_nmse = np.std(scores['test_nmse'])
        mean_r2 = np.mean(scores['test_r2'])
        std_r2 = np.std(scores['test_r2'])
        fit_time_mean = np.mean(scores['fit_time'])
        score_time_mean = np.mean(scores['score_time'])

        print(f"\n--- {model_name} ---")
        print(f"  RMSE: {mean_rmse:.4f} +/- {std_rmse:.4f}")
        print(f"  NMSE: {mean_nmse:.4f} +/- {std_nmse:.4f}")
        print(f"  R2 Score: {mean_r2:.4f} +/- {std_r2:.4f}")
        print(f"  Avg Fit Time: {fit_time_mean:.2f}s")
        print(f"  Avg Score Time: {score_time_mean:.2f}s")

        results_summary_list.append({
            'Model': model_name, 'Type': 'Baseline',
            'RMSE_mean': mean_rmse, 'RMSE_std': std_rmse,
            'NMSE_mean': mean_nmse, 'NMSE_std': std_nmse,
            'R2_mean': mean_r2, 'R2_std': std_r2,
            'FitTime': fit_time_mean
        })

# Create a DataFrame for easier comparison later
results_df = pd.DataFrame(results_summary_list)
print("\n--- Baseline Summary Table ---")
print(results_df.to_string(index=False, float_format="%.4f"))


# %% [markdown]
# **Interpretation:**
# - Compare Ridge, Lasso, and Random Forest. Lasso might perform feature selection implicitly (coefficients go to zero). Does it perform better/worse than Ridge or RF? How does its training time compare?
# - *Note:* Lasso's performance is highly dependent on `alpha`. The chosen value `0.01` might need tuning (e.g., using `LassoCV`).


# %% [markdown]
# ## 8. Feature Importance and Selection (Objective 4)
#
# Analyze feature importances for Random Forest and perform Sequential Feature Selection (SFS) using Ridge as the estimator.

# %%
# Feature Importances (for Random Forest) - Requires scaled features

# Create pipeline steps manually ONLY for this analysis
# Note: Re-extracting features here. Could optimize if memory is tight.
analysis_pipeline = Pipeline([
    ('feature_extraction', EMGFeatureExtractor()),
    ('scaling', StandardScaler())
])

# Fit and transform the full training data for analysis
with timer("Extracting & Scaling features for Importance/SFS Analysis"):
    X_features_scaled_for_analysis = analysis_pipeline.fit_transform(X_windowed, y_windowed)
    feature_names_analysis = analysis_pipeline.named_steps['feature_extraction'].get_feature_names_out()
    print(f"Shape of features for analysis: {X_features_scaled_for_analysis.shape}")

# Re-train the Random Forest model on these scaled features if needed for importance
# Or use the one trained inside the CV loop if accessible (less clean)
with timer("Training RandomForest on full data for Importance Analysis"):
    # Use the instance defined earlier
    model_rf.fit(X_features_scaled_for_analysis, y_windowed)

# Get and plot feature importances
importances = model_rf.feature_importances_
indices = np.argsort(importances)[::-1]
print("\nRandom Forest Feature Importances (Top 30):")
n_top_features = 30
for i in range(min(n_top_features, len(importances))):
    print(f"{i + 1}. Feature: {feature_names_analysis[indices[i]]} ({importances[indices[i]]:.4f})")

plt.figure(figsize=(12, 8))
plt.title("Random Forest Feature Importances (Top 30)")
plt.bar(range(min(n_top_features, len(importances))), importances[indices[:n_top_features]], align='center')
plt.xticks(range(min(n_top_features, len(importances))), [feature_names_analysis[i] for i in indices[:n_top_features]], rotation=90)
plt.xlim([-1, min(n_top_features, len(importances))]); plt.ylabel("Importance"); plt.tight_layout(); plt.show()

# %%
# Feature Selection using SFS
print("\n--- Performing Sequential Feature Selection (SFS) ---")
# Configure SFS
sfs = SequentialFeatureSelector(
    estimator=Ridge(alpha=1.0), # Use Ridge for speed
    n_features_to_select='auto',
    tol=1e-4, # Stop if score improvement is less than this (avoids selecting too many minor features)
    direction='forward',
    scoring=scorers['rmse'], # Minimize RMSE (maximize negative RMSE)
    cv=cv_strategy,
    n_jobs=-1
)

# Fit SFS using the scaled features and pass groups via metadata routing
with timer("Fitting Sequential Feature Selector"):
    sfs.fit(X_features_scaled_for_analysis, y_windowed, groups=groups)

# Get selected features
selected_feature_indices = sfs.get_support(indices=True)
selected_feature_names = [feature_names_analysis[i] for i in selected_feature_indices]
n_selected_features = len(selected_feature_names)

print(f"\nSFS selected {n_selected_features} features out of {len(feature_names_analysis)}.")
print("Selected Feature Names (first 50):")
print(selected_feature_names[:50]); print("..." if n_selected_features > 50 else "")

if n_selected_features > 0:
    selected_types = [f.split('_')[1] for f in selected_feature_names]
    selected_electrodes = [f.split('_')[0] for f in selected_feature_names]
    print("\nCounts of selected feature types:"); print(pd.Series(selected_types).value_counts())
    print("\nCounts of selected electrodes:"); print(pd.Series(selected_electrodes).value_counts())
else: print("\nWarning: SFS selected 0 features.")



# %% [markdown]
# ## 9. Final Baseline Pipeline with Selected Features (Optional Refinement)
#
# Create and evaluate a pipeline using only the features selected by SFS with the best baseline model. This is often done to see if selection improved performance or simplified the model effectively.

# %%
if n_selected_features > 0 and n_selected_features < len(feature_names_analysis):
    print("\n--- Evaluating Best Baseline Model with SFS Features ---")
    # Determine the best model from initial CV (e.g., based on lowest RMSE)
    # Make sure results_df is populated correctly
    if not results_df.empty:
        best_baseline_model_name = results_df.loc[results_df['RMSE_mean'].idxmin()]['Model']
        print(f"Selected best baseline model: {best_baseline_model_name}")

        if best_baseline_model_name == "Ridge":
            best_model_instance_sfs = Ridge(alpha=1.0)
        elif best_baseline_model_name == "Lasso":
             best_model_instance_sfs = Lasso(alpha=0.01, max_iter=2000, tol=1e-3)
        else: # RandomForest
            best_model_instance_sfs = RandomForestRegressor(
                n_estimators=100, max_depth=15, n_jobs=-1, random_state=42, max_features=0.33
            )

        # Get the base feature types selected by SFS
        selected_feature_types_only = sorted(list(set( f.split('_', 1)[1] for f in selected_feature_names)))
        print(f"\nUsing {len(selected_feature_types_only)} feature types selected by SFS: {selected_feature_types_only}")

        sfs_pipeline = Pipeline([
            ('feature_extraction', EMGFeatureExtractor(selected_features=selected_feature_types_only)),
            ('scaling', StandardScaler()),
            ('regression', best_model_instance_sfs)
        ])

        print("\nEvaluating pipeline with selected features:")
        with timer(f"Cross-validating {best_baseline_model_name} with {n_selected_features} SFS features"):
            sfs_scores = cross_validate(
                sfs_pipeline, X_windowed, y_windowed, cv=cv_strategy, scoring=scorers,
                n_jobs=-1, return_train_score=False, params={'groups': groups}
            )
            all_cv_results[f"{best_baseline_model_name}_SFS"] = sfs_scores # Store results
            model_name_sfs = f"{best_baseline_model_name}_SFS"

            # Add SFS results to summary list
            mean_rmse = -np.mean(sfs_scores['test_rmse']); std_rmse = np.std(sfs_scores['test_rmse'])
            mean_nmse = -np.mean(sfs_scores['test_nmse']); std_nmse = np.std(sfs_scores['test_nmse'])
            mean_r2 = np.mean(sfs_scores['test_r2']); std_r2 = np.std(sfs_scores['test_r2'])
            fit_time_mean = np.mean(sfs_scores['fit_time']); score_time_mean = np.mean(sfs_scores['score_time'])

            print(f"\n--- {model_name_sfs} ---")
            print(f"  RMSE: {mean_rmse:.4f} +/- {std_rmse:.4f}")
            print(f"  NMSE: {mean_nmse:.4f} +/- {std_nmse:.4f}")
            print(f"  R2 Score: {mean_r2:.4f} +/- {std_r2:.4f}")

            results_summary_list.append({
                'Model': model_name_sfs, 'Type': 'Baseline_SFS',
                'RMSE_mean': mean_rmse, 'RMSE_std': std_rmse, 'NMSE_mean': mean_nmse,
                'NMSE_std': std_nmse, 'R2_mean': mean_r2, 'R2_std': std_r2,
                'FitTime': fit_time_mean
            })
            # Update results_df
            results_df = pd.DataFrame(results_summary_list)

    else:
        print("Skipping SFS pipeline evaluation as baseline results are missing.")
else:
    print("\nSkipping evaluation with SFS features (SFS selected 0 or all features).")

# %% [markdown]
# ## 10. Sophisticated Approach: Neural Network (Objective 5)
#
# Implement a simple Multi-Layer Perceptron (MLP) using PyTorch for regression. We will perform cross-validation manually for the NN.

# %%
# Define the Neural Network Architecture
class EMG_NN(nn.Module):
    def __init__(self, n_features, n_joints, hidden_dim1=128, hidden_dim2=64, dropout_p=0.3):
        super(EMG_NN, self).__init__()
        self.layer_1 = nn.Linear(n_features, hidden_dim1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_p)
        self.layer_2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_p)
        self.output_layer = nn.Linear(hidden_dim2, n_joints)

    def forward(self, x):
        x = self.dropout1(self.relu1(self.layer_1(x)))
        x = self.dropout2(self.relu2(self.layer_2(x)))
        x = self.output_layer(x)
        return x

# %%
# Training and Evaluation Functions for PyTorch NN

def train_nn_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(dataloader.dataset)

def evaluate_nn_model(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * inputs.size(0)
            all_preds.append(outputs.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    avg_loss = running_loss / len(dataloader.dataset)
    predictions = np.concatenate(all_preds)
    true_values = np.concatenate(all_targets)
    return avg_loss, predictions, true_values

# %%
# Manual Cross-Validation Loop for Neural Network

# Hyperparameters
N_EPOCHS = 75 # Adjust as needed based on convergence
BATCH_SIZE = 64
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5 # L2 regularization for Adam

# NN requires features as input, we use the same extractor as baseline
# but scaling MUST happen INSIDE the CV loop to prevent data leakage
nn_feature_extractor = EMGFeatureExtractor() # Use all features for NN baseline

nn_cv_scores = {'test_rmse': [], 'test_nmse': [], 'test_r2': [], 'fit_time': [], 'score_time': []}

print("\n--- Evaluating Neural Network (Manual CV) ---")

# Use the originally extracted features (X_features) before scaling for analysis
# X_input_for_nn_cv = X_features
# Alternatively, re-extract features here if memory allows and X_features was overwritten
with timer("Re-extracting features for NN CV"):
     X_input_for_nn_cv = nn_feature_extractor.fit_transform(X_windowed)
     n_input_features = X_input_for_nn_cv.shape[1] # Get number of features
     print(f"NN Input feature shape: {X_input_for_nn_cv.shape}")


fold_num = 0
for train_idx, val_idx in cv_strategy.split(X_input_for_nn_cv, y_windowed, groups=groups):
    fold_num += 1
    print(f"\nStarting NN CV Fold {fold_num}/{N_SPLITS}")
    start_fold_time = time.time()

    # 1. Split data for this fold
    X_train_fold, X_val_fold = X_input_for_nn_cv[train_idx], X_input_for_nn_cv[val_idx]
    y_train_fold, y_val_fold = y_windowed[train_idx], y_windowed[val_idx]

    # 2. Scale features based *only* on training data for this fold
    scaler = StandardScaler()
    X_train_fold_scaled = scaler.fit_transform(X_train_fold)
    X_val_fold_scaled = scaler.transform(X_val_fold) # Use same scaler for validation

    # 3. Create PyTorch Datasets and DataLoaders
    train_dataset = TensorDataset(torch.FloatTensor(X_train_fold_scaled), torch.FloatTensor(y_train_fold))
    val_dataset = TensorDataset(torch.FloatTensor(X_val_fold_scaled), torch.FloatTensor(y_val_fold))

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # 4. Initialize model, loss, optimizer for this fold
    model_nn = EMG_NN(n_features=n_input_features, n_joints=N_JOINTS).to(device)
    criterion = nn.MSELoss() # Use MSE for regression
    optimizer = optim.Adam(model_nn.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    # 5. Training loop
    print(f"Fold {fold_num}: Training for {N_EPOCHS} epochs...")
    for epoch in range(N_EPOCHS):
        train_loss = train_nn_epoch(model_nn, train_loader, criterion, optimizer, device)
        # Optional: Evaluate validation loss per epoch for early stopping/monitoring
        if (epoch + 1) % 10 == 0 or epoch == N_EPOCHS - 1:
             val_loss_epoch, _, _ = evaluate_nn_model(model_nn, val_loader, criterion, device)
             print(f'  Epoch {epoch+1}/{N_EPOCHS}, Train Loss: {train_loss:.6f}, Val Loss: {val_loss_epoch:.6f}')
    fit_end_time = time.time()

    # 6. Evaluate on validation set
    print(f"Fold {fold_num}: Evaluating on validation set...")
    val_loss, y_pred_fold, y_true_fold = evaluate_nn_model(model_nn, val_loader, criterion, device)
    score_end_time = time.time()

    # 7. Calculate and store metrics for this fold
    rmse_fold = root_mean_squared_error(y_true_fold, y_pred_fold)
    nmse_fold = normalized_mean_squared_error(y_true_fold, y_pred_fold)
    r2_fold = r2_score(y_true_fold, y_pred_fold)

    nn_cv_scores['test_rmse'].append(-rmse_fold) # Store negative for consistency with scorers
    nn_cv_scores['test_nmse'].append(-nmse_fold)
    nn_cv_scores['test_r2'].append(r2_fold)
    nn_cv_scores['fit_time'].append(fit_end_time - start_fold_time)
    nn_cv_scores['score_time'].append(score_end_time - fit_end_time)

    print(f"Fold {fold_num} Metrics: RMSE={rmse_fold:.4f}, NMSE={nmse_fold:.4f}, R2={r2_fold:.4f}")

# Store overall NN results
all_cv_results["NeuralNetwork"] = nn_cv_scores

# %%
# Add NN results to the summary DataFrame
nn_model_name = "NeuralNetwork"
nn_scores = all_cv_results[nn_model_name]
mean_rmse = -np.mean(nn_scores['test_rmse'])
std_rmse = np.std(nn_scores['test_rmse'])
mean_nmse = -np.mean(nn_scores['test_nmse'])
std_nmse = np.std(nn_scores['test_nmse'])
mean_r2 = np.mean(nn_scores['test_r2'])
std_r2 = np.std(nn_scores['test_r2'])
fit_time_mean = np.mean(nn_scores['fit_time'])
score_time_mean = np.mean(nn_scores['score_time'])

results_summary_list.append({
    'Model': nn_model_name, 'Type': 'Sophisticated',
    'RMSE_mean': mean_rmse, 'RMSE_std': std_rmse,
    'NMSE_mean': mean_nmse, 'NMSE_std': std_nmse,
    'R2_mean': mean_r2, 'R2_std': std_r2,
    'FitTime': fit_time_mean
})

# Update and display the final comparison table
results_df = pd.DataFrame(results_summary_list)
print("\n--- Combined Model Comparison Table ---")
print(results_df.sort_values(by='RMSE_mean').to_string(index=False, float_format="%.4f"))


# %% [markdown]
# **Discussion of Neural Network Results (Objective 5):**
# - How does the Neural Network's performance (RMSE, NMSE, R2) compare to the baseline models (Ridge, Lasso, RF)?
# - Consider the training time (`FitTime`). NNs are typically much slower to train per fold.
# - Was the simple MLP architecture effective? Could it be improved (e.g., more layers, different activation functions, different optimizers, more epochs, learning rate scheduling)?
# - Did the NN generalize well across folds (look at the standard deviation of the metrics)? High variance might indicate overfitting or sensitivity to the specific session left out.
# - Document the chosen NN architecture, hyperparameters, and training procedure.


# %% [markdown]
# ## 11. Final Model Selection and Training (Prep for Objective 7)
#
# Based on the combined comparison table, choose the best overall model or ensemble (Objective 6 would go here if implemented). Then, train this chosen model on the *entire* training/validation dataset (`X_windowed`, `y_windowed` or `X_input_for_nn_cv` depending on the model) in preparation for predicting on the test set.

# %%
# Choose the best model based on cross-validation results (e.g., lowest RMSE)
best_overall_model_name = results_df.loc[results_df['RMSE_mean'].idxmin()]['Model']
print(f"\nSelected best overall model based on CV RMSE: {best_overall_model_name}")

# Prepare data and train the final selected model
final_trained_model = None

if best_overall_model_name == "NeuralNetwork":
    print("\nTraining final Neural Network on all data...")
    # 1. Scale all features using a scaler fitted on all data
    final_scaler_nn = StandardScaler()
    X_nn_scaled_final = final_scaler_nn.fit_transform(X_input_for_nn_cv) # Scale all features

    # 2. Create Dataset and DataLoader for all data
    final_dataset = TensorDataset(torch.FloatTensor(X_nn_scaled_final), torch.FloatTensor(y_windowed))
    # Use a larger batch size for final training if memory allows
    final_train_loader = DataLoader(final_dataset, batch_size=128, shuffle=True)

    # 3. Initialize final model, criterion, optimizer
    final_model_nn = EMG_NN(n_features=n_input_features, n_joints=N_JOINTS).to(device)
    final_criterion = nn.MSELoss()
    final_optimizer = optim.Adam(final_model_nn.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    # 4. Training loop (can potentially run for more epochs on final training)
    FINAL_N_EPOCHS = N_EPOCHS # Or increase slightly, e.g., N_EPOCHS + 20
    with timer(f"Training final NN for {FINAL_N_EPOCHS} epochs"):
        for epoch in range(FINAL_N_EPOCHS):
            train_loss = train_nn_epoch(final_model_nn, final_train_loader, final_criterion, final_optimizer, device)
            if (epoch + 1) % 10 == 0 or epoch == FINAL_N_EPOCHS - 1:
                print(f'  Epoch {epoch+1}/{FINAL_N_EPOCHS}, Train Loss: {train_loss:.6f}')

    final_trained_model = {'model': final_model_nn, 'scaler': final_scaler_nn} # Store model and scaler
    print("Final Neural Network trained.")

elif best_overall_model_name.endswith("_SFS"):
     print(f"\nTraining final SFS pipeline ({best_overall_model_name}) on all data...")
     # Recreate the SFS pipeline identified earlier
     base_model_name = best_overall_model_name.replace("_SFS","")
     if base_model_name == "Ridge": sfs_regressor = Ridge(alpha=1.0)
     elif base_model_name == "Lasso": sfs_regressor = Lasso(alpha=0.01, max_iter=2000, tol=1e-3)
     else: sfs_regressor = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42, max_features=0.33)

     # Ensure selected_feature_types_only is defined from SFS step
     if 'selected_feature_types_only' in locals():
          final_trained_pipeline_sfs = Pipeline([
               ('feature_extraction', EMGFeatureExtractor(selected_features=selected_feature_types_only)),
               ('scaling', StandardScaler()),
               ('regression', sfs_regressor)
          ])
          with timer("Fitting final SFS pipeline"):
               final_trained_pipeline_sfs.fit(X_windowed, y_windowed)
          final_trained_model = final_trained_pipeline_sfs # Store the pipeline
          print("Final SFS Pipeline trained.")
     else:
          print("Error: `selected_feature_types_only` not found. Cannot train SFS pipeline.")
          final_trained_model = None

else: # One of the standard baseline models (Ridge, Lasso, RF)
    print(f"\nTraining final baseline pipeline ({best_overall_model_name}) on all data...")
    # Get the model instance
    final_baseline_regressor = baseline_models_to_evaluate[best_overall_model_name]
    # Create pipeline with all features
    final_trained_pipeline_baseline = Pipeline([
            ('feature_extraction', EMGFeatureExtractor()), # Use all features
            ('scaling', StandardScaler()),
            ('regression', final_baseline_regressor)
        ])
    with timer("Fitting final baseline pipeline"):
        final_trained_pipeline_baseline.fit(X_windowed, y_windowed)
    final_trained_model = final_trained_pipeline_baseline # Store the pipeline
    print("Final Baseline Pipeline trained.")


# Saving the final model (optional but recommended)
if final_trained_model:
     if isinstance(final_trained_model, dict) and 'model' in final_trained_model: # NN Case
         # Save PyTorch model state and scaler
         torch.save(final_trained_model['model'].state_dict(), f'final_{best_overall_model_name}_model.pth')
         with open(f'final_{best_overall_model_name}_scaler.pkl', 'wb') as f:
             pickle.dump(final_trained_model['scaler'], f)
         print(f"Saved final NN model state and scaler for {best_overall_model_name}.")
     elif isinstance(final_trained_model, Pipeline): # Scikit-learn Pipeline Case
         with open(f'final_{best_overall_model_name}_pipeline.pkl', 'wb') as f:
             pickle.dump(final_trained_model, f)
         print(f"Saved final pipeline object for {best_overall_model_name}.")
     else:
          print("Final model type not recognized for saving.")


# %% [markdown]
# ---
# **Next Steps:** Objective 6 (Ensembling - if desired), Objective 7 (Prediction on Test Set using `final_trained_model`).
# ---
# Add cross_val_predict to imports if not already there
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LinearRegression # Potential meta-learner
# %% [markdown]
# ## 11. Objective 6: Ensembling Strategies
#
# Implement Averaging and Stacking ensembles using the out-of-fold predictions from the baseline models (Ridge, Lasso, RF) and the Neural Network.

# %% [markdown]
# ### 11.1 Generate Out-of-Fold (OOF) Predictions
#
# We need predictions for each data point made by a model that was *not* trained on that data point's fold.

# %%
oof_predictions = {} # Dictionary to store OOF predictions

# --- OOF Predictions for Scikit-learn Models ---
print("--- Generating OOF Predictions for Baseline Models ---")
# Use the pipelines defined earlier (or recreate them for clarity)
baseline_pipelines = {}
for model_name, model in baseline_models_to_evaluate.items():
     baseline_pipelines[model_name] = Pipeline([
          ('feature_extraction', EMGFeatureExtractor()), # Use all features for base models here
          ('scaling', StandardScaler()),
          ('regression', model)
     ])

# Use cross_val_predict
for model_name, pipeline in baseline_pipelines.items():
     with timer(f"Generating OOF predictions for {model_name}"):
          # Pass groups via params as metadata routing is enabled
          oof_preds = cross_val_predict(
               pipeline,
               X_windowed, # Use raw windowed sEMG
               y_windowed,
               cv=cv_strategy,
               n_jobs=-1,
               method='predict',
               params={'groups': groups} # Pass groups via params
          )
          oof_predictions[model_name] = oof_preds
     print(f"Shape of {model_name} OOF predictions: {oof_preds.shape}") # Should be (n_windows, n_joints)

# --- OOF Predictions for Neural Network ---
print("\n--- Generating OOF Predictions for Neural Network ---")
# We need to run the manual CV loop again, but this time focus on collecting predictions

# Ensure NN hyperparameters are defined
N_EPOCHS = 75 # Keep consistent with previous training
BATCH_SIZE = 64
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5

nn_oof_preds_list = [None] * len(X_input_for_nn_cv) # List to store predictions in correct order
nn_indices_list = [] # To verify order later if needed

# Re-use extracted features for NN (X_input_for_nn_cv)
if 'X_input_for_nn_cv' not in locals() or X_input_for_nn_cv is None:
     raise ValueError("NN features (X_input_for_nn_cv) not found. Re-run feature extraction for NN.")
n_input_features = X_input_for_nn_cv.shape[1]

fold_num = 0
with timer("Generating OOF predictions for Neural Network (Manual CV)"):
    for train_idx, val_idx in cv_strategy.split(X_input_for_nn_cv, y_windowed, groups=groups):
        fold_num += 1
        print(f"  Processing NN OOF Fold {fold_num}/{N_SPLITS}")
        start_fold_time = time.time()

        # 1. Split data (Features and Targets)
        X_train_fold, X_val_fold = X_input_for_nn_cv[train_idx], X_input_for_nn_cv[val_idx]
        y_train_fold, y_val_fold = y_windowed[train_idx], y_windowed[val_idx] # Targets needed for dataset

        # 2. Scale features based *only* on training data for this fold
        scaler = StandardScaler()
        X_train_fold_scaled = scaler.fit_transform(X_train_fold)
        X_val_fold_scaled = scaler.transform(X_val_fold)

        # 3. Create PyTorch Datasets and DataLoaders
        train_dataset = TensorDataset(torch.FloatTensor(X_train_fold_scaled), torch.FloatTensor(y_train_fold))
        val_dataset = TensorDataset(torch.FloatTensor(X_val_fold_scaled), torch.FloatTensor(y_val_fold)) # Need targets for loss calc
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

        # 4. Initialize model, loss, optimizer
        model_nn = EMG_NN(n_features=n_input_features, n_joints=N_JOINTS).to(device)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model_nn.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

        # 5. Training loop
        for epoch in range(N_EPOCHS): # Use same number of epochs as before
            _ = train_nn_epoch(model_nn, train_loader, criterion, optimizer, device) # Ignore train loss return value
            # Optional: print progress
            # if (epoch + 1) % 25 == 0: print(f"   Epoch {epoch+1}/{N_EPOCHS}")

        # 6. Evaluate and GET PREDICTIONS on validation set for this fold
        _, y_pred_fold, _ = evaluate_nn_model(model_nn, val_loader, criterion, device)

        # 7. Store predictions in the correct list positions using val_idx
        for i, idx in enumerate(val_idx):
            nn_oof_preds_list[idx] = y_pred_fold[i]
        nn_indices_list.extend(val_idx.tolist()) # Store indices to check order if needed

# Combine the list of predictions into a single numpy array
oof_predictions["NeuralNetwork"] = np.stack(nn_oof_preds_list)

# Verification (Optional): Check if all predictions were filled and order is correct
assert len(nn_indices_list) == len(X_input_for_nn_cv), "Number of NN OOF predictions doesn't match original data size."
# assert np.all(np.argsort(nn_indices_list) == np.arange(len(X_input_for_nn_cv))), "NN OOF predictions are not in the original order."
print(f"Shape of NN OOF predictions: {oof_predictions['NeuralNetwork'].shape}")

# %% [markdown]
# ### 11.2 Averaging Ensemble
#
# Compute the simple average of the OOF predictions from all base models.

# %%
print("\n--- Evaluating Averaging Ensemble ---")
model_names_for_avg = list(oof_predictions.keys()) # Should be ['Ridge', 'Lasso', 'RandomForest', 'NeuralNetwork']
print(f"Averaging predictions from models: {model_names_for_avg}")

# Summing up the predictions
oof_preds_sum = np.zeros_like(oof_predictions[model_names_for_avg[0]]) # Initialize with zeros shape
for model_name in model_names_for_avg:
    oof_preds_sum += oof_predictions[model_name]

# Calculating the average
oof_preds_avg = oof_preds_sum / len(model_names_for_avg)

# Evaluate the performance of the averaged predictions
avg_rmse = root_mean_squared_error(y_windowed, oof_preds_avg)
avg_nmse = normalized_mean_squared_error(y_windowed, oof_preds_avg)
avg_r2 = r2_score(y_windowed, oof_preds_avg)

print(f"\nAveraging Ensemble Performance (on OOF predictions):")
print(f"  RMSE: {avg_rmse:.4f}")
print(f"  NMSE: {avg_nmse:.4f}")
print(f"  R2 Score: {avg_r2:.4f}")

# Add results to summary
results_summary_list.append({
    'Model': 'AverageEnsemble', 'Type': 'Ensemble',
    'RMSE_mean': avg_rmse, 'RMSE_std': np.nan, # No std deviation from single OOF evaluation
    'NMSE_mean': avg_nmse, 'NMSE_std': np.nan,
    'R2_mean': avg_r2, 'R2_std': np.nan,
    'FitTime': np.nan # Fit time is combination of base models
})
# Update results_df
results_df = pd.DataFrame(results_summary_list)


# %% [markdown]
# ### 11.3 Stacking Ensemble (Meta-Learner)
#
# Train a secondary model (meta-learner) on the OOF predictions of the base models. We use Ridge as the meta-learner and evaluate it using cross-validation.

# %%
print("\n--- Evaluating Stacking Ensemble ---")
# 1. Create Meta-Features: Concatenate OOF predictions horizontally
model_names_for_stacking = list(oof_predictions.keys()) # Same models as averaging
print(f"Stacking predictions from models: {model_names_for_stacking}")

X_meta_list = [oof_predictions[model_name] for model_name in model_names_for_stacking]
X_meta = np.hstack(X_meta_list)

print(f"Shape of Meta-Features (X_meta): {X_meta.shape}") # Should be (n_windows, n_models * n_joints)

# 2. Choose and configure the Meta-Learner
# Using Ridge is a common and robust choice. Alpha might need tuning.
# LinearRegression could also be used: meta_model = LinearRegression(n_jobs=-1)
meta_model = Ridge(alpha=1.0)
print(f"Using Meta-Learner: {meta_model.__class__.__name__}")

# 3. Cross-Validate the Meta-Learner
# IMPORTANT: Use the same GroupKFold strategy on the meta-features
with timer(f"Cross-validating Stacking Meta-Learner ({meta_model.__class__.__name__})"):
     # No scaling needed for meta-features usually, predictions are somewhat comparable
     # If scales differ vastly, could add StandardScaler() here in a pipeline
     stacking_pipeline = Pipeline([
          # Optional: ('meta_scaling', StandardScaler()),
          ('meta_regression', meta_model)
     ])

     stacking_scores = cross_validate(
          stacking_pipeline,
          X_meta,           # Meta-features are input
          y_windowed,       # Original target
          cv=cv_strategy,
          scoring=scorers,
          n_jobs=-1,
          return_train_score=False,
          params={'groups': groups} # Pass groups via params
     )
     all_cv_results["StackingEnsemble"] = stacking_scores # Store results

# 4. Display Stacking CV Results
stacking_model_name = "StackingEnsemble"
mean_rmse = -np.mean(stacking_scores['test_rmse'])
std_rmse = np.std(stacking_scores['test_rmse'])
mean_nmse = -np.mean(stacking_scores['test_nmse'])
std_nmse = np.std(stacking_scores['test_nmse'])
mean_r2 = np.mean(stacking_scores['test_r2'])
std_r2 = np.std(stacking_scores['test_r2'])
fit_time_mean = np.mean(stacking_scores['fit_time'])
score_time_mean = np.mean(stacking_scores['score_time'])

print(f"\nStacking Ensemble Performance (Cross-Validated):")
print(f"  RMSE: {mean_rmse:.4f} +/- {std_rmse:.4f}")
print(f"  NMSE: {mean_nmse:.4f} +/- {std_nmse:.4f}")
print(f"  R2 Score: {mean_r2:.4f} +/- {std_r2:.4f}")
print(f"  Avg Meta Fit Time: {fit_time_mean:.2f}s")

# Add results to summary
results_summary_list.append({
    'Model': stacking_model_name, 'Type': 'Ensemble',
    'RMSE_mean': mean_rmse, 'RMSE_std': std_rmse,
    'NMSE_mean': mean_nmse, 'NMSE_std': std_nmse,
    'R2_mean': mean_r2, 'R2_std': std_r2,
    'FitTime': fit_time_mean # Fit time of the meta-learner
})
# Update results_df
results_df = pd.DataFrame(results_summary_list)
# %% [markdown]
# ### 11.4 Meta-Learner Contribution Analysis (Stacking)
#
# Train the meta-learner on *all* the OOF predictions and examine its coefficients to infer base model contributions.

# %%
print("\n--- Stacking Meta-Learner Contribution Analysis ---")
# Train the chosen meta-model on all OOF data
final_meta_pipeline = Pipeline([('meta_regression', Ridge(alpha=1.0))]) # Use the same Ridge
final_meta_pipeline.fit(X_meta, y_windowed)

# Get the trained meta-regressor
meta_regressor_trained = final_meta_pipeline.named_steps['meta_regression']

# Examine coefficients (for linear meta-learners like Ridge)
# Shape of coef_ is (n_targets, n_meta_features) = (51, num_models * 51)
coefficients = meta_regressor_trained.coef_
print(f"Shape of meta-learner coefficients: {coefficients.shape}")

# Calculate average absolute coefficient magnitude for each base model's block of predictions
n_models = len(model_names_for_stacking)
n_joints = y_windowed.shape[1] # Should be 51
avg_coef_magnitudes = {}

for i, model_name in enumerate(model_names_for_stacking):
    start_col = i * n_joints
    end_col = (i + 1) * n_joints
    # Get the block of coefficients corresponding to this model's predictions
    model_coefs = coefficients[:, start_col:end_col]
    # Calculate the average absolute magnitude across all targets and all features for this model
    avg_mag = np.mean(np.abs(model_coefs))
    avg_coef_magnitudes[model_name] = avg_mag
    print(f"  Avg. Abs. Coef Magnitude for {model_name}: {avg_mag:.4f}")

# Create a Series for easy sorting/viewing
coef_mag_series = pd.Series(avg_coef_magnitudes).sort_values(ascending=False)
print("\nRelative Contribution based on Avg. Abs. Coefficient Magnitude:")
print(coef_mag_series)

# Plotting the contributions
plt.figure(figsize=(8, 4))
coef_mag_series.plot(kind='bar')
plt.title('Stacking Meta-Learner: Base Model Contribution (Avg. Abs. Coef Mag)')
plt.ylabel('Average Absolute Coefficient Magnitude')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


# %% [markdown]
# **Discussion of Ensembling Results (Objective 6):**
#
# 1.  **Performance Comparison:**
#     *   Did the Averaging ensemble outperform the *best* individual base model? Did it outperform the *average* base model? Usually, averaging reduces variance and performs well if base models have similar performance levels and their errors are somewhat uncorrelated.
#     *   Did the Stacking ensemble outperform Averaging? Did it outperform the best base model? Stacking has the potential to be better if the meta-learner can effectively learn how to combine the base predictions, weighting better models higher or exploiting situations where one model is better than others. However, it also has more risk of overfitting, especially if the meta-learner is complex or if the OOF predictions used for training it are noisy.
#     *   Compare the RMSE, NMSE, and R2 scores across all individual models and both ensembles using the final comparison table.
#
# 2.  **Bias-Variance Tradeoff:**
#     *   Individual models (especially complex ones like RF or NN if not well-regularized) might have lower bias but higher variance (performance fluctuates more depending on the training data/fold).
#     *   Averaging primarily reduces variance by averaging out the noise in individual predictions. The bias remains roughly the average bias of the base models. If base models are very biased, the average will also be biased.
#     *   Stacking aims to reduce bias (by learning optimal combinations) and potentially variance (if the meta-learner is simple and stable, like Ridge). If the meta-learner overfits the OOF predictions, it might increase variance.
#     *   Relate the observed performance gains (or lack thereof) of the ensembles to these concepts. Did the ensembles seem to stabilize predictions (lower std dev in CV if calculated, or generally better average performance)?
#
# 3.  **Meta-Learner Contributions (Stacking):**
#     *   Which base models did the meta-learner seem to rely on most, based on the coefficient magnitude analysis? Does this align with the individual performance of those base models?
#     *   Sometimes a meta-learner might give high weight to a seemingly weaker model if that model provides complementary information (i.e., makes correct predictions where other models fail).
#     *   *Caveat:* Coefficient magnitude is an imperfect proxy for importance, especially if the input meta-features (OOF predictions) have different scales. Standardizing `X_meta` before fitting the meta-learner could make magnitudes more comparable but wasn't done here for simplicity.
#
# *Document your specific findings and interpretations based on your results in the notebook.*


# %% [markdown]
# ## 12. Combined Model Comparison Table (Updated)
#
# Display the final table including baseline, SFS (if run), NN, and ensemble results.

# %%
print("\n--- Final Combined Model Comparison Table ---")
# Ensure results_df is updated
results_df = pd.DataFrame(results_summary_list)
print(results_df.sort_values(by='RMSE_mean').to_string(index=False, float_format="%.4f"))

# %% [markdown]
# ---
# **Now, proceed to the modified Final Model Selection cell.**
# ---